### Decision Trees - C4.5

converting numerical into categories

In [4]:
import pandas as pd
import numpy as np
import math

data = pd.DataFrame({
    'outlook': ['s','s','o','r','r','r','o','s','s','r','s','o','o','r'],
    'temp': [85,80,83,70,68,65,64,72,69,75,75,72,81,71],
    'humidity': [85,90,78,96,80,70,65,95,70,80,70,90,75,80],
    'wind': ['w','s','w','w','w','s','s','w','w','w','s','s','w','s'],
    'play': [0,0,1,1,1,0,1,0,1,1,1,1,1,0]
})

# 2. Convert numerical attributes into categories
data['temp'] = np.where(data['temp'] > 75, 'h', 'l')
data['humidity'] = np.where(data['humidity'] > 80, 'h', 'l')

features = ['outlook', 'temp', 'humidity', 'wind']
target = 'play'


def entropy(feature):
    c1 = np.sum(feature==0) / len(feature)
    c2 = np.sum(feature==1) / len(feature)
    if c1 > 0 and c2 > 0 :
        r = c1 * math.log2(c1) + c2 * math.log2(c2)
    elif c1 > 0 :
        r = c1 * math.log2(c1)
    else :
        r = c2 * math.log2(c2)
    return -r


def information_gain(data, feature):
    total_entropy = entropy(data["play"])
    weighted_entropy = 0
    split_info = 0
    total = len(data)

    for value in data[feature].unique():
        subset = data[data[feature] == value]
        p = len(subset) / total

        weighted_entropy += p * entropy(subset["play"])

        if p > 0:
            split_info -= p * math.log2(p)

    gain = total_entropy - weighted_entropy

    if split_info == 0:
        return 0

    return gain / split_info


def build_tree(df, features, target):

    # If all target values are identical
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    # If no features remain, return the majority class
    if len(features) == 0:
        return df[target].mode().iloc[0]

    gains = {}
    for feature in features :
        gains[feature] = information_gain(df, feature)

    best_feature = max(gains, key=gains.get)

    tree = {best_feature: {}}

    # Create branches for each feature value
    for value, subset in df.groupby(best_feature):
        remaining_features = [f for f in features if f != best_feature]
        subset = subset.drop(columns=[best_feature])

        tree[best_feature][value] = build_tree(subset, remaining_features, target)

    return tree


# 6. Classify a new sample
def classify(tree, sample, default_class):

    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    value = sample[feature]

    if value in tree[feature]:
        return classify(tree[feature][value], sample, default_class)

    return default_class


# 7. Build and print the decision tree
tree = build_tree(data, features, target)

print("Generated C4.5 Decision Tree:")
from pprint import pprint
pprint(tree,sort_dicts=False)


# 8. Classify a new sample
new_sample = pd.Series({
    'outlook': 's',
    'temp': 'l',
    'humidity': 'l',
    'wind': 'w'
})

default_class = data[target].mode().iloc[0]

prediction = classify(tree, new_sample, default_class)

if prediction == 1:
    print("Classification: Play Tennis = Yes")
else:
    print("Classification: Play Tennis = No")


Generated C4.5 Decision Tree:
{'outlook': {'o': np.int64(1),
             'r': {'wind': {'s': np.int64(0), 'w': np.int64(1)}},
             's': {'humidity': {'h': np.int64(0), 'l': np.int64(1)}}}}
Classification: Play Tennis = Yes


In [11]:
from collections import Counter
from math import log2

# Dataset
ROWS = [
    {"Day": 1, "Outlook": "Sunny", "Temp": 85, "Humidity": 85, "Wind": "Weak", "Decision": "No"},
    {"Day": 2, "Outlook": "Sunny", "Temp": 80, "Humidity": 90, "Wind": "Strong", "Decision": "No"},
    {"Day": 3, "Outlook": "Overcast", "Temp": 83, "Humidity": 78, "Wind": "Weak", "Decision": "Yes"},
    {"Day": 4, "Outlook": "Rain", "Temp": 70, "Humidity": 96, "Wind": "Weak", "Decision": "Yes"},
    {"Day": 5, "Outlook": "Rain", "Temp": 68, "Humidity": 80, "Wind": "Weak", "Decision": "Yes"},
    {"Day": 6, "Outlook": "Rain", "Temp": 65, "Humidity": 70, "Wind": "Strong", "Decision": "No"},
    {"Day": 7, "Outlook": "Overcast", "Temp": 64, "Humidity": 65, "Wind": "Strong", "Decision": "Yes"},
    {"Day": 8, "Outlook": "Sunny", "Temp": 72, "Humidity": 95, "Wind": "Weak", "Decision": "No"},
    {"Day": 9, "Outlook": "Sunny", "Temp": 69, "Humidity": 70, "Wind": "Weak", "Decision": "Yes"},
    {"Day": 10, "Outlook": "Rain", "Temp": 75, "Humidity": 80, "Wind": "Weak", "Decision": "Yes"},
    {"Day": 11, "Outlook": "Sunny", "Temp": 75, "Humidity": 70, "Wind": "Strong", "Decision": "Yes"},
    {"Day": 12, "Outlook": "Overcast", "Temp": 72, "Humidity": 90, "Wind": "Strong", "Decision": "Yes"},
    {"Day": 13, "Outlook": "Overcast", "Temp": 81, "Humidity": 75, "Wind": "Weak", "Decision": "Yes"},
    {"Day": 14, "Outlook": "Rain", "Temp": 71, "Humidity": 80, "Wind": "Strong", "Decision": "No"}
]

FEATURES = ["Outlook", "Temp", "Humidity", "Wind"]
NUMERIC = ["Temp", "Humidity"]


def entropy(rows):
    counts = Counter(row["Decision"] for row in rows)
    total = len(rows)
    result = 0

    for count in counts.values():
        probability = count / total
        result -= probability * log2(probability)

    return result


def split_rows(rows, feature, threshold=None):
    groups = {}

    for row in rows:
        if threshold is None:
            branch = row[feature]
        elif row[feature] <= threshold:
            branch = "<="
        else:
            branch = ">"

        if branch not in groups:
            groups[branch] = []

        groups[branch].append(row)

    return groups


def calculate_scores(rows, groups):
    total = len(rows)
    weighted_entropy = 0
    split_information = 0

    for group in groups.values():
        fraction = len(group) / total
        weighted_entropy += fraction * entropy(group)
        split_information -= fraction * log2(fraction)

    gain = entropy(rows) - weighted_entropy

    if split_information == 0:
        gain_ratio = 0
    else:
        gain_ratio = gain / split_information

    return gain, gain_ratio


def find_best_split(rows, available_features):
    choices = []

    for feature in available_features:
        # Categorical features have one split.
        # Numeric features try thresholds between consecutive values.
        if feature in NUMERIC:
            values = sorted(set(row[feature] for row in rows))
            thresholds = []

            for i in range(len(values) - 1):
                threshold = (values[i] + values[i + 1]) / 2
                thresholds.append(threshold)
        else:
            thresholds = [None]

        best_for_feature = None

        # Choose this feature's best threshold by information gain.
        for threshold in thresholds:
            groups = split_rows(rows, feature, threshold)

            if len(groups) < 2:
                continue

            gain, ratio = calculate_scores(rows, groups)
            option = (feature, threshold, groups, gain, ratio)

            if best_for_feature is None:
                best_for_feature = option
            elif gain > best_for_feature[3]:
                best_for_feature = option
            elif gain == best_for_feature[3] and ratio > best_for_feature[4]:
                best_for_feature = option

        if best_for_feature is not None:
            choices.append(best_for_feature)

    if not choices:
        return None

    # Keep features whose gain is at least the average gain.
    average_gain = sum(option[3] for option in choices) / len(choices)

    best = None

    for option in choices:
        gain = option[3]
        ratio = option[4]

        if gain > 0 and gain >= average_gain - 1e-12:
            if best is None or ratio > best[4]:
                best = option

    return best


def build_tree(rows, available_features):
    counts = Counter(row["Decision"] for row in rows)

    if counts["Yes"] >= counts["No"]:
        majority = "Yes"
    else:
        majority = "No"

    # Stop if the group is pure or no features remain.
    if len(counts) == 1 or not available_features:
        return majority

    best = find_best_split(rows, available_features)

    if best is None:
        return majority

    feature, threshold, groups, gain, ratio = best

    tree = {
        "feature": feature,
        "threshold": threshold,
        "fallback": majority,
        "children": {}
    }

    # A categorical feature is used once.
    # A numeric feature may be split again at another threshold.
    if feature in NUMERIC:
        remaining = available_features
    else:
        remaining = []

        for item in available_features:
            if item != feature:
                remaining.append(item)

    for branch, group in groups.items():
        tree["children"][branch] = build_tree(group, remaining)

    return tree


def predict(tree, sample):
    current = tree

    while isinstance(current, dict):
        feature = current["feature"]
        threshold = current["threshold"]

        if feature not in sample:
            return current["fallback"]

        if threshold is None:
            branch = sample[feature]
        elif sample[feature] <= threshold:
            branch = "<="
        else:
            branch = ">"

        if branch not in current["children"]:
            return current["fallback"]

        current = current["children"][branch]

    return current



# Build and print the tree
tree = build_tree(ROWS, FEATURES)

print("Decision Tree:")
pprint(tree, sort_dicts=False)

# Predict a new day
new_day = {
    "Outlook": "Sunny",
    "Temp": 74,
    "Humidity": 70,
    "Wind": "Weak"
}

print("\nNew day:", new_day)
print("Predicted decision:", predict(tree, new_day))

# Check predictions on the provided data
correct = 0

for row in ROWS:
    if predict(tree, row) == row["Decision"]:
        correct += 1

print("Training rows classified correctly:", correct, "/", len(ROWS))

Decision Tree:
{'feature': 'Outlook',
 'threshold': None,
 'fallback': 'Yes',
 'children': {'Sunny': {'feature': 'Humidity',
                        'threshold': 77.5,
                        'fallback': 'No',
                        'children': {'>': 'No', '<=': 'Yes'}},
              'Overcast': 'Yes',
              'Rain': {'feature': 'Wind',
                       'threshold': None,
                       'fallback': 'Yes',
                       'children': {'Weak': 'Yes', 'Strong': 'No'}}}}

New day: {'Outlook': 'Sunny', 'Temp': 74, 'Humidity': 70, 'Wind': 'Weak'}
Predicted decision: Yes
Training rows classified correctly: 14 / 14
